In [1]:
library(tximport)
library(DESeq2)
library(readr)
library(dplyr)
library(tibble)
library(pheatmap)
library(ggplot2)

# Directorio de trabajo
base_dir <- "C:/Users/fran_/Documents/Doctorado/Inicios/Dani/GSE244241"

Loading required package: S4Vectors

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min



Attaching package: 'S4Vectors'


The following object is masked from 'package:utils':

    findMatches


The follo

In [2]:
# 1. Metadatos de las muestras
samples <- tibble(
  sample = c("SRR26204549","SRR26204550","SRR26204553","SRR26204554","SRR26204557","SRR26204558"),
  pair = c("P1","P1","P2","P2","P3","P3"),
  cell_type = factor(c("MIM","HBC","MIM","HBC","MIM","HBC"), levels=c("HBC","MIM"))
)

# 2. Búsqueda automática de archivos Salmon
# Buscamos todos los quant.sf en cualquier subcarpeta dentro de base_dir
all_quants <- list.files(base_dir, pattern = "quant\\.sf$", recursive = TRUE, full.names = TRUE)

# Le asignamos a cada muestra su ruta correspondiente
files <- sapply(samples$sample, function(s) {
  # Buscamos el archivo que coincida con el nombre de la muestra
  match <- grep(paste0("/", s, "/quant\\.sf$"), all_quants, value = TRUE)
  return(match[1]) 
})
names(files) <- samples$sample

# Control para verificar que R encontró todo
if(any(is.na(files))) {
  warning("Faltan archivos quant.sf para: ", paste(names(files)[is.na(files)], collapse=", "))
} else {
  cat("¡Todos los archivos quant.sf fueron encontrados con éxito!\n")
}

# 3. Leer GTF y mapear transcriptos a genes
gtf <- read_tsv(file.path(base_dir,"gencode.v44.annotation.gtf.gz"), comment="#", col_names=FALSE)
colnames(gtf) <- c("chr","source","feature","start","end","score","strand","frame","attr")

extract_attr <- function(x, key) sub(paste0('.*',key,' "([^"]+)".*'),"\\1",x)

tx2gene <- gtf %>% 
  filter(feature == "transcript") %>%
  transmute(
    TXNAME = sub("\\..*","", extract_attr(attr,"transcript_id")),
    GENEID = sub("\\..*","", extract_attr(attr,"gene_id")),
    GENENAME = extract_attr(attr,"gene_name")
  )

¡Todos los archivos quant.sf fueron encontrados con éxito!


Rows: 3424189 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (7): X1, X2, X3, X6, X7, X8, X9
dbl (2): X4, X5

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [3]:
# 1. Importar a tximport usando los archivos que encontramos
txi <- tximport(files, type="salmon", tx2gene=tx2gene[,c("TXNAME","GENEID")], ignoreTxVersion=TRUE)

# 2. Extraer la matriz de conteos crudos
raw_counts <- txi$counts

# 3. Guardarla en un CSV por seguridad/consultas futuras
write.csv(
  as.data.frame(raw_counts), 
  file.path(base_dir, "raw_counts_salmon.csv")
)

# Un pantallazo para verificar que cargó bien
cat("Dimensiones de la matriz de conteos:", dim(raw_counts), "\n")
head(raw_counts)

reading in files with read_tsv

1 
2 
3 
4 
5 
6 


summarizing abundance

summarizing counts

summarizing length



Dimensiones de la matriz de conteos: 62266 6 


,SRR26204549,SRR26204550,SRR26204553,SRR26204554,SRR26204557,SRR26204558
ENSG00000000003,58.537,27.755,59.391,17.487,92.241,64.119
ENSG00000000005,0.000,0.000,0.000,0.000,0.000,0.000
ENSG00000000419,621.717,352.612,627.993,482.901,727.311,237.941
ENSG00000000457,415.009,145.818,349.843,150.651,378.222,102.122
ENSG00000000460,48.303,51.010,111.924,60.847,90.008,54.475
ENSG00000000938,374.000,7658.001,7481.001,6140.001,5676.000,1924.998


In [4]:
# 1. Quitar los decimales de los Ensembl IDs
raw_ids <- sub("\\..*", "", rownames(raw_counts))

# 2. Armar el diccionario Ensembl -> Symbol usando el tx2gene
gene_map <- tx2gene %>% distinct(GENEID, GENENAME)

# 3. Mapear los nombres a la matriz
raw_symbols <- gene_map$GENENAME[match(raw_ids, gene_map$GENEID)]

# 4. Filtrar filas que no tengan símbolo válido
keep_raw <- !is.na(raw_symbols) & raw_symbols != ""
df_raw_filtered <- as.data.frame(raw_counts[keep_raw, ])
df_raw_filtered$gene_symbol <- raw_symbols[keep_raw]

# 5. Colapsar símbolos duplicados SUMANDO los conteos (clave para datos crudos)
df_raw_collapsed <- df_raw_filtered %>%
  group_by(gene_symbol) %>%
  summarise(across(everything(), sum), .groups = "drop")

# 6. Reconstruir la matriz final
mat_raw_symbol <- as.matrix(df_raw_collapsed[, -1])
rownames(mat_raw_symbol) <- df_raw_collapsed$gene_symbol

# 7. Exportar el CSV
write.csv(
  as.data.frame(mat_raw_symbol), 
  file.path(base_dir, "raw_counts_salmon_SYMBOLS.csv")
)

cat("Genes totales en matriz cruda con símbolos:", nrow(mat_raw_symbol), "\n")

Genes totales en matriz cruda con símbolos: 60883 


In [5]:
# 1. Renombrar las columnas de la matriz normalizada (por si no corriste lo anterior)
nuevos_nombres <- c("Mim 1", "Hbc 1", "Mim 2", "Hbc 2", "Mim 3", "Hbc 3")
colnames(mat_gene) <- nuevos_nombres

# 2. REORDENAR las columnas para agrupar Mim con Mim y Hbc con Hbc
orden_deseado <- c("Mim 1", "Mim 2", "Mim 3", "Hbc 1", "Hbc 2", "Hbc 3")
mat_gene <- mat_gene[, orden_deseado]

# 3. Filtrar genes de sexado
sex_genes <- c("KDM6A","KDM5C","EIF2S3","DDX3X","SMC1A","USP9X","ZFX",
               "UTY","KDM5D","DDX3Y","ZFY","SRY","SOX9","AMH","DMRT1","XIST")

sex_mat <- mat_gene[rownames(mat_gene) %in% sex_genes, , drop = FALSE]

# Limpiar genes sin variación
sex_mat <- sex_mat[apply(sex_mat, 1, function(x) all(is.finite(x)) && stats::sd(x, na.rm=TRUE) > 0), , drop = FALSE]

# 4. Graficar y guardar
if (nrow(sex_mat) > 0) {
  # Mostrar en el notebook
  pheatmap(sex_mat, scale="row", cluster_cols=FALSE, cluster_rows=TRUE, main="Marcadores de Sexo (X e Y)")
  
  # Guardar en PDF
  pdf(file.path(base_dir, "heatmap_sex.pdf"))
  pheatmap(sex_mat, scale="row", cluster_cols=FALSE, cluster_rows=TRUE, main="Marcadores de Sexo (X e Y)")
  dev.off()
  
} else {
  warning("No se encontraron marcadores de sexo con expresión.")
}

ERROR: Error: object 'mat_gene' not found


In [ ]:
# 1. Filtrar los metadatos para quedarnos solo con las HBC
samples_hbc <- samples %>% filter(cell_type == "HBC")

# 2. Asignar el grupo de sexo CORRECTO según lo que vimos en el heatmap
# (HBC 1 y 2 = Masculino, HBC 3 = Femenino)
samples_hbc$sex <- c("Male", "Male", "Female")

# Convertir a factor indicando el nivel de referencia (vamos a ver qué cambia en Female respecto a Male)
samples_hbc$sex <- factor(samples_hbc$sex, levels = c("Male", "Female"))

# 3. Filtrar los archivos de Salmon para que solo cargue estas 3 muestras
files_hbc <- files[samples_hbc$sample]

# 4. Importar datos exclusivos de HBC
txi_hbc <- tximport(files_hbc, type="salmon", tx2gene=tx2gene[,c("TXNAME","GENEID")], ignoreTxVersion=TRUE)

# 5. Crear objeto DESeq2 con el nuevo diseño experimental (~sex)
dds_hbc <- DESeqDataSetFromTximport(txi_hbc, colData=as.data.frame(samples_hbc), design=~sex)

# 6. Correr el análisis diferencial
dds_hbc <- DESeq(dds_hbc)

# 7. Extraer los resultados (Female vs Male) y guardar
res_hbc <- results(dds_hbc, contrast=c("sex", "Female", "Male"))

write.csv(
  as.data.frame(res_hbc), 
  file.path(base_dir, "DESeq2_HBC_Female_vs_Male.csv")
)

cat("¡Análisis terminado con los sexos corregidos! Resultados guardados en 'DESeq2_HBC_Female_vs_Male.csv'.\n")

reading in files with read_tsv

1 
2 
3 


summarizing abundance

summarizing counts

summarizing length

using counts and average transcript lengths from tximport

estimating size factors

using 'avgTxLength' from assays(dds), correcting for library size

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



¡Análisis terminado con los sexos corregidos! Resultados guardados en 'DESeq2_HBC_Female_vs_Male.csv'.


In [ ]:
# 1. Cargar el archivo que generamos recién (o usar el objeto 'res_hbc' si sigue en memoria)
res_df <- as.data.frame(res_hbc)

# 2. Limpiar los IDs (quitar versiones si las hay) y preparar el mapeo
res_df$GENEID <- sub("\\..*", "", rownames(res_df))
gene_map <- tx2gene %>% distinct(GENEID, GENENAME)

# 3. Cruzar los datos para obtener el Gene Symbol
res_symbol <- res_df %>%
  left_join(gene_map, by = "GENEID") %>%
  select(GENEID, GENENAME, everything()) # Reordenar columnas para que el nombre esté al principio

# 4. Guardar el archivo final ya traducido
write.csv(
  res_symbol, 
  file.path(base_dir, "DESeq2_HBC_Female_vs_Male_SYMBOLS.csv"),
  row.names = FALSE
)

# 5. Ver los genes más significativos (Top 10 por p-value)
cat("Top 10 genes con mayor diferencia por sexo:\n")
print(head(res_symbol %>% arrange(pvalue), 10))

Top 10 genes con mayor diferencia por sexo:
            GENEID        GENENAME   baseMean log2FoldChange     lfcSE
1  ENSG00000225972        MTND1P23 15535.8847       9.758929 0.5315386
2  ENSG00000262902        MTCO1P40  5753.4285       9.413959 0.5562012
3  ENSG00000256148 ENSG00000256148  1398.7391       9.433185 0.6573157
4  ENSG00000101670            LIPG  1206.8520       7.640177 0.5549763
5  ENSG00000067048           DDX3Y  2061.8720      -7.565113 0.6394923
6  ENSG00000166147            FBN1  3660.2611       6.136782 0.5277148
7  ENSG00000065618         COL17A1  1616.6343       6.083696 0.5269581
8  ENSG00000100234           TIMP3 28449.7201       6.570609 0.5788040
9  ENSG00000168077          SCARA3   937.6142       7.083010 0.6285108
10 ENSG00000146674          IGFBP3  4319.1582       5.391951 0.4932401
        stat       pvalue         padj
1   18.35977 2.757495e-75 4.747856e-71
2   16.92546 2.920750e-64 2.514473e-60
3   14.35107 1.049033e-46 6.020750e-43
4   13.76667 4.0440

In [ ]:
# 1. Cargar los datos con símbolos que ya tenemos
res_all <- read_csv(file.path(base_dir, "DESeq2_HBC_Female_vs_Male_SYMBOLS.csv"))

# 2. Aplicar filtros: significancia y magnitud de cambio
# Filtramos padj < 0.05 y |log2FoldChange| > 1
degs_filtered <- res_all %>%
  filter(padj < 0.05) %>%
  filter(abs(log2FoldChange) > 1) %>%
  # Nos quedamos solo con la columna del Symbol (y el LFC por si querés saber la dirección)
  select(GENENAME, log2FoldChange, padj) %>%
  arrange(desc(log2FoldChange)) # Los de arriba son más altos en la hembra (HBC 3)

# 3. Guardar la lista limpia
write.csv(
  degs_filtered, 
  file.path(base_dir, "HBC_Female_vs_Male_ONLY_DEGS.csv"),
  row.names = FALSE
)

# 4. Mostrar resumen en pantalla
cat("Análisis de DEGs finalizado.\n")
cat("Total de genes diferenciales encontrados:", nrow(degs_filtered), "\n")
cat("\nPrimeros 5 genes con mayor expresión en HEMBRA (LFC > 0):\n")
print(head(degs_filtered, 5))
cat("\nPrimeros 5 genes con mayor expresión en MACHOS (LFC < 0):\n")
print(tail(degs_filtered, 5))

Rows: 62266 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (2): GENEID, GENENAME
dbl (6): baseMean, log2FoldChange, lfcSE, stat, pvalue, padj

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Análisis de DEGs finalizado.
Total de genes diferenciales encontrados: 1354 

Primeros 5 genes con mayor expresión en HEMBRA (LFC > 0):
# A tibble: 5 × 3
  GENENAME        log2FoldChange     padj
  <chr>                    <dbl>    <dbl>
1 FAM43B                   10.5  6.94e- 8
2 FMC1-LUC7L2              10.4  9.19e- 8
3 ENSG00000260978          10.2  3.76e- 7
4 HERC3                    10.1  7.52e- 7
5 MTND1P23                  9.76 4.75e-71

Primeros 5 genes con mayor expresión en MACHOS (LFC < 0):
# A tibble: 5 × 3
  GENENAME        log2FoldChange      padj
  <chr>                    <dbl>     <dbl>
1 ENSG00000289733          -9.18 0.00134  
2 EIF3FP3                  -9.84 0.000186 
3 TXLNGY                  -10.3  0.0000732
4 PRKY                    -11.1  0.0000153
5 ENSG00000267645         -21.9  0.00520  


## Ahora con MIMN

In [ ]:
# 1. Filtrar los metadatos para quedarnos solo con las muestras MIM
samples_mim <- samples %>% filter(cell_type == "MIM")

# 2. Asignar el sexo (Mim 1 y 2 = Male, Mim 3 = Female)
samples_mim$sex <- c("Male", "Male", "Female")
samples_mim$sex <- factor(samples_mim$sex, levels = c("Male", "Female"))

# 3. Filtrar archivos de Salmon para MIM
files_mim <- files[samples_mim$sample]

# 4. Importar datos
txi_mim <- tximport(files_mim, type="salmon", tx2gene=tx2gene[,c("TXNAME","GENEID")], ignoreTxVersion=TRUE)

# 5. Crear objeto DESeq2 (~sex)
dds_mim <- DESeqDataSetFromTximport(txi_mim, colData=as.data.frame(samples_mim), design=~sex)

# 6. Correr DESeq2
dds_mim <- DESeq(dds_mim)

# 7. Extraer resultados y mapear a Symbols
res_mim <- results(dds_mim, contrast=c("sex", "Female", "Male"))
res_mim_df <- as.data.frame(res_mim)
res_mim_df$GENEID <- sub("\\..*", "", rownames(res_mim_df))

# 8. Unir con nombres de genes y filtrar DEGs (|LFC| > 1 y padj < 0.05)
res_mim_symbol <- res_mim_df %>%
  left_join(distinct(tx2gene, GENEID, GENENAME), by = "GENEID") %>%
  select(GENENAME, log2FoldChange, padj, everything())

degs_mim <- res_mim_symbol %>%
  filter(padj < 0.05 & abs(log2FoldChange) > 1) %>%
  arrange(desc(log2FoldChange))

# 9. Guardar archivos
write.csv(res_mim_symbol, file.path(base_dir, "DESeq2_MIM_Female_vs_Male_ALL.csv"), row.names = FALSE)
write.csv(degs_mim, file.path(base_dir, "MIM_Female_vs_Male_ONLY_DEGS.csv"), row.names = FALSE)

cat("Análisis MIM terminado.\n")
cat("Total de DEGs en MIM:", nrow(degs_mim), "\n")

reading in files with read_tsv

1 
2 
3 


summarizing abundance

summarizing counts

summarizing length

using counts and average transcript lengths from tximport

estimating size factors

using 'avgTxLength' from assays(dds), correcting for library size

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing



Análisis MIM terminado.
Total de DEGs en MIM: 15 


## Para multi

In [ ]:
# 2. Leer tus datos
counts <- read.csv("MULTI_gene_expression_raw_counts.csv", row.names = 1, check.names = FALSE)

# Importante: DESeq2 necesita enteros. Tus datos tienen decimales.
counts_int <- round(as.matrix(counts))

# 3. Crear el diseño experimental (Metadata)
# El orden debe coincidir con las columnas de tu CSV: 51, 52, 55, 56, 59, 60
info_muestras <- data.frame(
  row.names = colnames(counts_int),
  TipoCelular = c("Monocito", "HBC", "Monocito", "HBC", "Monocito", "HBC"),
  Sexo = c("Masculino", "Femenino", "Femenino", "Femenino", "Masculino", "Masculino")
)

# 4. Análisis para MONOCITOS (MIM)
# Filtramos la matriz para quedarnos solo con las columnas de Monocitos
counts_mono <- counts_int[, info_muestras$TipoCelular == "Monocito"]
meta_mono <- info_muestras[info_muestras$TipoCelular == "Monocito", ]

dds_mono <- DESeqDataSetFromMatrix(countData = counts_mono,
                                   colData = meta_mono,
                                   design = ~ Sexo)

# Correr DESeq
dds_mono <- DESeq(dds_mono)
res_mono <- results(dds_mono, contrast = c("Sexo", "Femenino", "Masculino"))

# 5. Análisis para HBC
counts_hbc <- counts_int[, info_muestras$TipoCelular == "HBC"]
meta_hbc <- info_muestras[info_muestras$TipoCelular == "HBC", ]

dds_hbc <- DESeqDataSetFromMatrix(countData = counts_hbc,
                                  colData = meta_hbc,
                                  design = ~ Sexo)

dds_hbc <- DESeq(dds_hbc)
res_hbc <- results(dds_hbc, contrast = c("Sexo", "Femenino", "Masculino"))

# 6. Guardar resultados
write.csv(as.data.frame(res_mono), "ResultadosMULTI_Monocitos_Fem_vs_Masc.csv")
write.csv(as.data.frame(res_hbc), "ResultadosMULTI_HBC_Fem_vs_Masc.csv")

# Tip: Para ver los genes más significativos rápido
summary(res_mono)
head(res_mono[order(res_mono$padj), ])

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
"some variables in design formula are characters, converting to factors"
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

converting counts to integer mode

Warning message in DESeqDataSet(se, design = design, ignoreRank):
"some variables in design formula are characters, converting to factors"
estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing




out of 34844 with nonzero total read count
adjusted p-value < 0.1
LFC > 0 (up)       : 820, 2.4%
LFC < 0 (down)     : 656, 1.9%
outliers [1]       : 0, 0%
low counts [2]     : 20118, 58%
(mean count < 28)
[1] see 'cooksCutoff' argument of ?results
[2] see 'independentFiltering' argument of ?results



log2 fold change (MLE): Sexo Femenino vs Masculino 
Wald test p-value: Sexo Femenino vs Masculino 
DataFrame with 6 rows and 6 columns
        baseMean log2FoldChange     lfcSE      stat      pvalue        padj
       <numeric>      <numeric> <numeric> <numeric>   <numeric>   <numeric>
OLFM4   2942.926        7.96345  0.551072   14.4508 2.47675e-47 3.64726e-43
MMP8    4458.449        7.52786  0.536743   14.0251 1.09480e-44 8.06100e-41
RPS4Y1  1757.530       -8.19328  0.613799  -13.3485 1.20902e-40 5.93467e-37
TCN1     963.506        6.36912  0.485625   13.1153 2.69112e-39 9.90736e-36
ADGRG3   665.303        6.46094  0.506755   12.7496 3.13201e-37 9.22440e-34
PADI4   1127.368        5.59210  0.442204   12.6460 1.17759e-36 2.89020e-33

In [ ]:
# --- FILTRADO PARA MONOCITOS ---
# 1. Convertir a data frame y filtrar
degs_mono <- as.data.frame(res_mono) %>%
  filter(padj < 0.05 & abs(log2FoldChange) > 1) %>%
  arrange(padj) # Ordenar por los más significativos

# 2. Guardar solo si hay resultados
write.csv(degs_mono, "MULTI_DEGs_Monocitos_Fem_vs_Masc.csv")


# --- FILTRADO PARA HBC ---
# 1. Convertir a data frame y filtrar
degs_hbc <- as.data.frame(res_hbc) %>%
  filter(padj < 0.05 & abs(log2FoldChange) > 1) %>%
  arrange(padj)

# 2. Guardar
write.csv(degs_hbc, "MULTI_DEGs_HBC_Fem_vs_Masc.csv")

# --- RESUMEN EN CONSOLA ---
cat("Resumen de DEGs encontrados:\n")
cat("Monocitos:", nrow(degs_mono), "genes.\n")
cat("HBC:", nrow(degs_hbc), "genes.\n")

Resumen de DEGs encontrados:
Monocitos: 1040 genes.
HBC: 1321 genes.


# Multi vs Monogravida

In [ ]:
library(DESeq2)
library(dplyr)

# --- PASO 1: Unificar Conteos ---
# Nota: Asegúrate de que las filas (genes) sean las mismas en ambos archivos.
# Si usas los 'counts_int' (redondeados) de Multi y los de Primigrávida:

counts_primi <- ... (los conteos que usaste para el análisis de Mimi/HBC primi)
counts_multi <- counts_int

# Unimos por columnas (muestras)
all_counts <- cbind(counts_primi, counts_multi)

# --- PASO 2: Crear la Metadata Global ---
# Debemos definir Sexo y también "Estado" (Primi vs Multi) para controlar ese efecto
metadata_global <- data.frame(
  row.names = colnames(all_counts),
  Sexo = c(rep("Masculino", 2), "Femenino",  # Ejemplo Primi MIM
           rep("Masculino", 2), "Femenino",  # Ejemplo Primi HBC
           "Masculino", "Femenino", "Femenino", "Femenino", "Masculino", "Masculino"), # Multi
  Condicion = c(rep("Primigravida", 6), rep("Multigravida", 6))
)

# Aseguramos que los factores tengan el nivel de referencia correcto
metadata_global$Sexo <- factor(metadata_global$Sexo, levels = c("Masculino", "Femenino"))
metadata_global$Condicion <- factor(metadata_global$Condicion)

ERROR: Error: object 'counts_primi' not found
